# NLP Exercises

We have five exercises in this section. The exercises are:
1. Build your own tokenizer, where you need to implement two functions to implement a tokenizer based on regular expression.
2. Get tags from Trump speech.
3. Get the nouns in the last 10 sentences from Trump's speech and find the nouns divided by sentencens. Use SpaCy.
4. Build your own Bag Of Words implementation using tokenizer created before.
5. Build a 5-gram model and clean up the results.

## Exercise 1. Build your own tokenizer

Build two different tokenizers:
- ``tokenize_sentence``: function tokenizing text into sentences,
- ``tokenize_word``: function tokenizing text into words.

In [16]:
import re
from typing import List

def tokenize_words(text: str) -> list:
    
    """Tokenize text into words using regex.

    Parameters
    ----------
    text: str
            Text to be tokenized

    Returns
    -------
    List[str]
            List containing words tokenized from text

    """
    return re.findall(r"\b[\w']+\b", text)

def tokenize_sentence(text: str) -> list:
    """Tokenize text into words using regex.

    Parameters
    ----------
    text: str
            Text to be tokenized

    Returns
    -------
    List[str]
            List containing words tokenized from text

    """
    sentences = re.split(r'(?<=[.!?])\s*(?=[A-Z])', text)
    return [s.strip() for s in sentences if s]

text = "Here we go again. I was supposed to add this text later.\
Well, it's 10.p.m. here, and I'm actually having fun making this course. :o\
I hope you are getting along fine with this presentation, I really did try.\
And one last sentence, just so you can test you tokenizers better."

print("Tokenized sentences:")
print(tokenize_sentence(text))

print("Tokenized words:")
print(tokenize_words(text))

Tokenized sentences:
['Here we go again.', 'I was supposed to add this text later.', "Well, it's 10.p.m. here, and I'm actually having fun making this course. :oI hope you are getting along fine with this presentation, I really did try.", 'And one last sentence, just so you can test you tokenizers better.']
Tokenized words:
['Here', 'we', 'go', 'again', 'I', 'was', 'supposed', 'to', 'add', 'this', 'text', 'later', 'Well', "it's", '10', 'p', 'm', 'here', 'and', "I'm", 'actually', 'having', 'fun', 'making', 'this', 'course', 'oI', 'hope', 'you', 'are', 'getting', 'along', 'fine', 'with', 'this', 'presentation', 'I', 'really', 'did', 'try', 'And', 'one', 'last', 'sentence', 'just', 'so', 'you', 'can', 'test', 'you', 'tokenizers', 'better']


## Exercise 2. Get tags from Trump speech using NLTK

You should use the ``trump.txt`` file, read it and find the tags for each word. Use NLTK for it.

In [17]:


import nltk
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag

# You may need to download these resources if they aren't already on your machine
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# Open and read the file
file = open("../trump.txt", "r", encoding="utf-8") 
trump = file.read()

# Step 1: Tokenize the text into words
words = word_tokenize(trump)

# Step 2: Get POS tags for each word
tags = pos_tag(words)

# Print the first 10 tags to verify
print(tags[:10])

# Close the file
file.close()

[('Thank', 'NNP'), ('you', 'PRP'), ('very', 'RB'), ('much', 'RB'), ('.', '.'), ('Mr.', 'NNP'), ('Speaker', 'NNP'), (',', ','), ('Mr.', 'NNP'), ('Vice', 'NNP')]


[nltk_data] Downloading package punkt to /Users/jans/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/jans/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## Exercise 3. Get the nouns in the last 10 sentences from Trump's speech and find the nouns divided by sentencens. Use SpaCy.

Please use Python list features to get the last 10 sentences and display nouns from it.

In [18]:
import spacy

nlp = spacy.load("en_core_web_sm")

file = open("../trump.txt", "r", encoding='utf-8') 
trump = file.read() 

doc = nlp(trump)

last_10_sentences = list(doc.sents)[-10:]

print(f"--- Extracting Nouns from the Last 10 Sentences ---\n")

for i, sent in enumerate(last_10_sentences, 1):
    nouns = [token.text for token in sent if token.pos_ in ["NOUN", "PROPN"]]
    
    print(f"Sentence {i}: {sent.text.strip()}")
    print(f"Nouns found: {nouns}")
    print("-" * 30)

file.close()

--- Extracting Nouns from the Last 10 Sentences ---

Sentence 1: When we fulfill this vision, when we celebrate our 250 years of glorious freedom, we will look back on tonight as when this new chapter of American greatness began.
Nouns found: ['vision', 'years', 'freedom', 'tonight', 'chapter', 'greatness']
------------------------------
Sentence 2: The time for small thinking is over.
Nouns found: ['time', 'thinking']
------------------------------
Sentence 3: The time for trivial fights is behind us.
Nouns found: ['time', 'fights']
------------------------------
Sentence 4: We just need the courage to share the dreams that fill our hearts, the bravery to express the hopes that stir our souls, and the confidence to turn those hopes and those dreams into action.
Nouns found: ['courage', 'dreams', 'hearts', 'bravery', 'hopes', 'souls', 'confidence', 'hopes', 'dreams', 'action']
------------------------------
Sentence 5: From now on, America will be empowered by our aspirations, not burd

## Exercise 4. Build your own Bag Of Words implementation using tokenizer created before 

You need to implement following methods:

- ``fit_transform`` - gets a list of strings and returns matrix with it's BoW representation
- ``get_features_names`` - returns list of words corresponding to columns in BoW

In [19]:
import numpy as np
import spacy

class BagOfWords:

    def __init__(self):
        self.__nlp = spacy.load("en_core_web_sm")
        self.__feature_names = []
    
    def __tokenize(self, text: str) -> list:
        doc = self.__nlp(text.lower())
        return [token.text for token in doc if not token.is_punct and not token.is_space]
    
    def fit_transform(self, corpus: list) -> np.array:
        tokenized_corpus = [self.__tokenize(doc) for doc in corpus]
        
        vocabulary = sorted(list(set(word for doc in tokenized_corpus for word in doc)))
        self.__feature_names = vocabulary
        
        matrix = np.zeros((len(corpus), len(vocabulary)), dtype=int)
        
        word_to_index = {word: i for i, word in enumerate(vocabulary)}
        
        for doc_idx, doc_tokens in enumerate(tokenized_corpus):
            for word in doc_tokens:
                if word in word_to_index:
                    word_idx = word_to_index[word]
                    matrix[doc_idx, word_idx] += 1
                    
        return matrix

    def get_feature_names(self) -> list:
        return self.__feature_names

corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]    
    
vectorizer = BagOfWords()

X = vectorizer.fit_transform(corpus)

print("BoW Matrix:")
print(X)

print("\nFeature Names (Vocabulary):")
print(vectorizer.get_feature_names())

print(f"\nVocabulary Length: {len(vectorizer.get_feature_names())}")

BoW Matrix:
[[0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1]
 [1 0 0 0 2 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 2 1 0 0 1 0 1 0 0]]

Feature Names (Vocabulary):
['a', 'as', 'bag', 'based', 'below', 'can', 'counting', 'document', 'documents', 'gives', 'is', 'matrix', 'most', 'multiple', 'occur', 'occurences', 'of', 'on', 'once', 'only', 'pretty', 'really', 'see', 'sparse', 'the', 'third', 'this', 'throughout', 'us', 'words', 'you']

Vocabulary Length: 31


## Exercise 5. Build a 5-gram model and clean up the results.

There are three tasks to do:
1. Use 5-gram model instead of 3.
2. Change to capital letter each first letter of a sentence.
3. Remove the whitespace between the last word in a sentence and . ! or ?.

Hint: for 2. and 3. implement a function called ``clean_generated()`` that takes the generated text and fix both issues at once. It could be easier to fix the text after it's generated rather then doing some changes in the while loop.

In [20]:
import random
import re
from nltk.book import *

wall_street = text7.tokens
tokens = wall_street

def cleanup():
    compiled_pattern = re.compile("^[a-zA-Z0-9.!?]")
    clean = list(filter(compiled_pattern.match, tokens))
    return clean

tokens = cleanup()

N = 5 
SEP = " "
sentence_count = 5

def build_ngrams():
    ngrams = []
    for i in range(len(tokens) - N + 1):
        ngrams.append(tokens[i:i + N])
    return ngrams

def ngram_freqs(ngrams):
    counts = {}
    for ngram in ngrams:
        token_seq = SEP.join(ngram[:-1])
        last_token = ngram[-1]
        if token_seq not in counts:
            counts[token_seq] = {}
        if last_token not in counts[token_seq]:
            counts[token_seq][last_token] = 0
        counts[token_seq][last_token] += 1
    return counts

def next_word(text, N, counts):
    token_seq = SEP.join(text.split()[-(N-1):])
    if token_seq not in counts:
        return random.choice(list(counts.keys())).split()[0]
    
    choices = counts[token_seq].items()
    total = sum(weight for choice, weight in choices)
    r = random.uniform(0, total)
    upto = 0
    for choice, weight in choices:
        upto += weight
        if upto > r: return choice
    return random.choice(list(counts.keys())).split()[0]

def clean_generated(text):
    
    text = re.sub(r'\s+([.!?])', r'\1', text)
    
    text = text[0].upper() + text[1:] if text else text
    
    text = re.sub(r'([.!?]\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), text)
    
    return text

ngrams = build_ngrams()
counts = ngram_freqs(ngrams)

start_seq = "We have to be"
generated = start_seq.lower()

sentences = 0
max_words = 100 
word_count = 0

while sentences < sentence_count and word_count < max_words:
    new_word = next_word(generated, N, counts)
    generated += SEP + new_word
    word_count += 1
    if new_word in ['.', '!', '?']:
        sentences += 1

final_text = clean_generated(generated)
print(final_text)

We have to be more like the department store not the boutique. IRAs. SHAREDATA Inc. Said 0 it will sell a stake in the chain to management and take other steps to reduce its investment in retailing. Equitable of Iowa Cos. Des Moines had been seeking a buyer for the 36-store Younkers chain since June when it announced its intention to free up capital to expand its insurance business. But Equitable said 0 it was seeking offers for its five radio stations in order to concentrate on its programming business.
